# Code 8.5.1: Temperature, top-k, and top-p

Defines `sample_next`, which applies temperature, top-k, and top-p to a vector of logits and samples one token (temperature 0 means greedy decoding), and checks it on a toy vocabulary of five tokens. Save it as `sampling.py`; the other listings import it.


In [ ]:
import torch

def sample_next(logits, temperature=1.0, top_k=None, top_p=None, generator=None):
    """Choose the next token from a (V,) vector of logits.
    temperature=0 means greedy; otherwise rescale, truncate, renormalize, and sample."""
    if temperature == 0:
        return int(logits.argmax())
    logits = logits / temperature                           # a new tensor; the caller's is untouched
    if top_k is not None:                                   # keep the k largest logits
        kth = torch.topk(logits, top_k).values[-1]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p is not None:                                   # keep the smallest set with mass >= p
        probs, order = logits.softmax(-1).sort(descending=True)
        drop = probs.cumsum(-1) - probs >= top_p            # the mass before this token already reaches p
        logits[order[drop]] = float("-inf")
    return int(torch.multinomial(logits.softmax(-1), 1, generator=generator))

if __name__ == "__main__":
    logits = torch.tensor([3.0, 2.0, 1.0, 0.5, -1.0])       # a toy vocabulary of five tokens
    for tau in [0.5, 1.0, 2.0]:
        print(f"tau={tau}:", (logits / tau).softmax(-1).numpy().round(3))
    g = torch.Generator().manual_seed(0)
    for name, kw in [("top-k 2", dict(top_k=2)), ("top-p 0.9", dict(top_p=0.9))]:
        draws = torch.tensor([sample_next(logits, generator=g, **kw) for _ in range(10000)])
        print(f"{name}: sampled frequencies", (torch.bincount(draws, minlength=5) / 10000).numpy().round(3))
# tau=0.5: [0.862 0.117 0.016 0.006 0.   ]
# tau=1.0: [0.624 0.229 0.084 0.051 0.011]
# tau=2.0: [0.417 0.253 0.154 0.12  0.056]
# top-k 2: sampled frequencies [0.734 0.266 0.    0.    0.   ]
# top-p 0.9: sampled frequencies [0.666 0.246 0.088 0.    0.   ]
